As of now we are building software based on rule based logics like if-else,for,while to take decisions, but if we can able to call LLM through API then we can bring AI based Human like intelligence for decision Making.

Why Do we Need LangChain?

Every LLM provide came up with his own code, LangChain is a abstraction Framework where we can use different model with minimal code changes. It is LangChain ecosystem and bulitin Plugins which can easy AI based Model Building.

# API Key Loading

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


# Model Invokation

In [12]:
from langchain.chat_models import init_chat_model

llm_call=init_chat_model(model="groq:openai/gpt-oss-120b")
res=llm_call.invoke("Hello, how are you?")
res.content


"Hello! I'm doing well, thank you. How can I assist you today?"

# Messages and Context Setup

In [15]:
from langchain.messages import HumanMessage,SystemMessage

Context_prompt=SystemMessage(content="You are a Mainframe Subject Matter Expert. Answer the questions")
User_prompt=HumanMessage(content="Explain about comp, comp2, comp3?")

full_prompt=[Context_prompt,User_prompt]

res=llm_call.invoke(full_prompt)

print(res.content,end="",flush=True)

## Overview  

In IBM z/OS (and most other IBM‑compatible) COBOL implementations the **`COMP`**, **`COMP‑2`**, and **`COMP‑3`** clauses are the three most‑commonly‑used “binary/packed” storage formats.  

| Clause | Common name | Physical representation | Typical size (bytes) | Typical range | Typical use |
|--------|-------------|--------------------------|----------------------|----------------|--------------|
| `COMP` (or `COMP‑5`) | Binary (native) | Two’s‑complement integer, machine‑dependent byte order & alignment | 1, 2, 4, or 8 (depends on picture) | Signed integer up to 2ⁿ⁻¹‑1 (where *n* = bits) | Arithmetic‑intensive fields (counters, indexes, amounts that fit in an integer) |
| `COMP‑2` | Double‑precision floating‑point | IEEE‑754 64‑bit floating point (binary64) | 8 | Approx. ±1.7E‑308 … ±1.7E+308, 15‑16 decimal digits of precision | Scientific/engineering calculations where fractional values and a large dynamic range are required |
| `COMP‑3` | Packed‑decimal (BCD) | Two d


We can see LLM are providing response which is a free form, but for applications we need LLMS to provide structured output, our application may have multiple LLM and this Output will be passed to other LLMs as input.

We have Pydantic Lib which can be used to get Structured response, Validation, Parsing, Conversion

# Structured Response with Pydantic

In [ ]:
from pydantic import BaseModel,Field
from typing import List

class Mainframe_Bot(BaseModel):
    """ 
    Schema for Structured Response for the Mainframe Bot
    """
    response:str=Field(description="Response to the User Query")
    examples:List[str]=Field(description="List of Examples")

In [18]:
structured_llm=llm_call.with_structured_output(Mainframe_Bot)

res=structured_llm.invoke(full_prompt)

print(res.response)
print(res.examples)

In COBOL (the most common language on IBM mainframes) the **USAGE** clause controls how a numeric data item is stored in memory. The three most frequently‑used USAGE options are **COMP**, **COMP‑2**, and **COMP‑3**.  

### 1. COMP (binary)
* **What it is** – A binary (integer) representation. The compiler decides the exact size (usually 2, 4, or 8 bytes) based on the picture clause.
* **Typical picture** – `PIC S9(4) COMP` (2‑byte binary), `PIC S9(9) COMP` (4‑byte binary), `PIC S9(18) COMP` (8‑byte binary).
* **Range** – Depends on the number of bytes: 2‑byte signed binary ranges from –32,768 to +32,767; 4‑byte from –2,147,483,648 to +2,147,483,647; 8‑byte from –9,223,372,036,854,775,808 to +9,223,372,036,854,775,807.
* **Performance** – Fastest arithmetic because the CPU works directly on binary integers. No conversion is needed for most arithmetic instructions.
* **When to use** – When you need integer arithmetic, counters, indexes, or when the numeric range fits comfortably within t

# Memory, Agents, Middleware

We can append the AI response to prompt and can create Memory but we have inbuilt function in Langchain for memory.

Agent = Model + System Prompt +Tool Calling + Memory

In [42]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

mem=InMemorySaver()
Sum_mid= SummarizationMiddleware(model=llm_call,trigger={"messages":10},keep_recent = 3)

Mainframe_agent = create_agent(
    model=llm_call,
    system_prompt=Context_prompt,
    checkpointer= mem,
    middleware=[Sum_mid]
)

agent_conf={"configurable":{"thread_id":"Mainframe_bot"}}

res = Mainframe_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Explain about COMP, COMP-2, and COMP-3?"
        }
    ]
},config=agent_conf,checkpointer=mem)

print(res["messages"][-1].content)

**COBOL Computational (COMP) data items**  
In COBOL the *computational* (or *binary*) picture clauses are used when you want the compiler to store a numeric field in a compact binary form rather than as packed or display characters.  
There are three main computational types that you will see in most COBOL implementations:

| Type | COBOL name | Typical usage | Storage format | Size (bytes) | Range (signed) | Typical picture clause |
|------|------------|---------------|----------------|--------------|----------------|------------------------|
| **COMP** | `COMP` (or `BINARY` on IBM‑compatible systems) | General‑purpose binary arithmetic, often for counters, indexes, totals, or any numeric value that will be used in arithmetic expressions. | Two’s‑complement binary integer (no decimal point) | 1, 2, 4 or 8 bytes (implementation‑dependent; most IBM‑z/OS and OpenCOBOL use 2‑byte for PIC 9(4), 4‑byte for PIC 9(9) and 8‑byte for larger) | –2ⁿ⁻¹ … +2ⁿ⁻¹‑1 (where *n* = number of bits) | `PI

# Tool Integration

LLM comes with Knowledge Cutoff, so if we ingegrate LLM with Tools then LLM, then LLM can call tool when required

These tolls can be custom tools(@tool def) or inbuilt tools

In [47]:
from langchain_community.tools import DuckDuckGoSearchRun

websearch=DuckDuckGoSearchRun()

In [50]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

mem = InMemorySaver()

Sum_mid = SummarizationMiddleware(
    model=llm_call,
    trigger={"messages": 10},
    keep_recent=3
)

Mainframe_agent = create_agent(
    model=llm_call,
    system_prompt=Context_prompt,
    checkpointer=mem,
    middleware=[Sum_mid],
    tools=[websearch]
)

agent_conf = {
    "configurable": {
        "thread_id": "Mainframe_bot"
    }
}

res = Mainframe_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather Today in Palakonda?"
            }
        ]
    },
    config=agent_conf
)

print(res["messages"][-1].content)

DDGSException: DNSError: DNSError('error sending request for url (https://wt.wikipedia.org/w/api.php?action=opensearch&profile=fuzzy&limit=1&search=Palakonda%20weather.com) > error resolving DNS > DNS error: no records found for Query { name: Name("wt.wikipedia.org."), query_type: A, query_class: IN }')